---
layout: post
assignment: true
courses: { csa: {week: 3} }
categories: [Java, Object-References]
lesson_language: Java
lesson_topic: Object-References
lesson_part: interactive
lesson_type: lesson
toc: true
codemirror: true
title: Methods Passing and Returning References of an Object
menu: nav/csa_units/csaunit3.html
permalink: /csa/unit_03/3_6
---

Java always passes arguments **by value** - but for an object, the value being copied is a *reference*, not the object itself. That one fact explains why mutating a parameter can change the caller's data, while reassigning it can't.

## Learning Targets

- How object references are passed into methods
- What happens when you mutate an object passed as a parameter
- Why reassigning a parameter inside a method does not change the caller's object
- How to write methods that return object references

---

## Vocabulary

| Term | Meaning | Example |
| --- | --- | --- |
| **Reference** | The address of an object in memory - what a variable actually stores | `Shape s = new Shape(...);` - `s` holds a reference, not the object |
| **Pass-by-value** | Java copies the value of every argument into the method's parameter | Primitives copy the number; objects copy the reference |
| **Mutate** | Call a method that changes a field on the object a reference points to | `s.set_width(10);` |
| **Reassign** | Point a variable at a completely different object | `s = new Shape(...);` |
| **Aliasing** | Two variables holding the same reference, so both see one shared object | `Shape b = a;` |

## Passing Objects into Methods

This is **Topic 5.6, "Writing Methods,"** under Learning Objective MOD-2.F (College Board, 2020). Passing a reference parameter makes the formal and actual parameters aliases of the same object - Oracle's documentation confirms the same rule: the reference itself is passed by value, but the object's fields can still be changed through it (Oracle, n.d.).

We'll keep using the `Shape` class from the last lesson, with getters/setters and an area calculation added in.


In [ ]:
// CODE_RUNNER: Press Run to compile the shared Shape class this lesson builds on, then watch it construct and print a sample shape.

class Shape {
    protected String name;
    private int length;
    private int width;

    public Shape(String name, int length, int width) {
        this.name = name;
        this.length = length;
        this.width = width;
    }

    public String get_name() { return this.name; }
    public int get_length() { return this.length; }
    public int get_width() { return this.width; }

    public void set_name(String n) { this.name = n; }
    public void set_length(int l) { this.length = l; }
    public void set_width(int w) { this.width = w; }

    public double calc_area() {
        return this.length * this.width;
    }

    public void print_shape() {
        System.out.println(this.name + ": length=" + this.length + " width=" + this.width);
    }
}

public class ShapeDemo {
    public static void main(String[] args) {
        Shape sample = new Shape("sample", 6, 3);
        sample.print_shape();
        System.out.println("area=" + sample.calc_area());
    }
}


### A. Simple - Mutating Through a Parameter

In [ ]:
// CODE_RUNNER: Press Run and watch doubleWidth mutate the Shape through its parameter -- rect sees the change once the method returns.

class Shape {
    protected String name;
    private int length;
    private int width;

    public Shape(String name, int length, int width) {
        this.name = name;
        this.length = length;
        this.width = width;
    }

    public int get_width() { return this.width; }
    public void set_width(int w) { this.width = w; }

    public void print_shape() {
        System.out.println(this.name + ": length=" + this.length + " width=" + this.width);
    }
}

public class MutateDemo {
    public static void doubleWidth(Shape s) {
        s.set_width(s.get_width() * 2);
    }

    public static void main(String[] args) {
        Shape rect = new Shape("rectangle", 10, 5);
        rect.print_shape();

        doubleWidth(rect);
        rect.print_shape();
    }
}


<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

    rectangle: length=10 width=5
    rectangle: length=10 width=10

Calling `doubleWidth(rect)` copies the *reference* stored in `rect` into the parameter `s` -- `s` and `rect` now point at the exact same `Shape` object, they are aliases. Inside the method, `s.set_width(s.get_width() * 2)` calls a setter on that shared object, changing its `width` field from 5 to 10. Because there is only one `Shape` involved (never a copy of the object itself), the change is visible through `rect` the moment `doubleWidth` returns -- no `return` value is needed.

</div>
</details>


### B. Intermediate - Reassigning Does NOT Escape the Method

In [ ]:
// CODE_RUNNER: Press Run and see that reassigning s inside replaceShape never escapes the method -- rect keeps pointing at the original Shape.

class Shape {
    protected String name;
    private int length;
    private int width;

    public Shape(String name, int length, int width) {
        this.name = name;
        this.length = length;
        this.width = width;
    }

    public void print_shape() {
        System.out.println(this.name + ": length=" + this.length + " width=" + this.width);
    }
}

public class ReassignDemo {
    public static void replaceShape(Shape s) {
        s = new Shape("replacement", 1, 1); // only reassigns the local copy of the reference
    }

    public static void main(String[] args) {
        Shape rect = new Shape("rectangle", 10, 5);
        replaceShape(rect);
        rect.print_shape();
    }
}


<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

    rectangle: length=10 width=5

`replaceShape(rect)` receives a *copy of the reference* in its parameter `s`. The line `s = new Shape("replacement", 1, 1);` does not hand a new object back to the caller -- it only makes the local copy `s` point somewhere new. `rect` in `main` is a completely separate variable holding its own copy of the original reference, so it never sees the reassignment. Reassigning a parameter can never redirect the caller's variable, because parameters are passed by value (the value being the reference itself) -- only the local copy changes.

</div>
</details>


> [!IMPORTANT]
> Mutate a field → caller sees it. Reassign the variable → caller doesn't. Confusing these two is one of the most common bugs when working with objects.


## Popcorn Hack 1

> [!TIP]
> Write a method `growShape(Shape s)` that adds 5 to both the length and width of the `Shape` it's given. Call it on a `Shape`, print before and after, and confirm the change persists outside the method.

In [ ]:
// CODE_RUNNER: Fill in growShape so it adds 5 to both length and width, then hit Run to check the output.

class Shape {
    protected String name;
    private int length, width;
    public Shape(String name, int length, int width) { this.name = name; this.length = length; this.width = width; }
    public int get_length() { return length; }
    public int get_width() { return width; }
    public void set_length(int l) { length = l; }
    public void set_width(int w) { width = w; }
    public void print_shape() { System.out.println(name + ": length=" + length + " width=" + width); }
}

public class Main {
    // TODO: add 5 to both s's length and s's width
    public static void growShape(Shape s) {

    }

    public static void main(String[] args) {
        Shape rect = new Shape("rectangle", 10, 5);
        rect.print_shape();
        growShape(rect);
        rect.print_shape(); // expect: length=15 width=10
    }
}


<details>
<summary>Need a hint?</summary>

Use the getters and setters already on `Shape` - `growShape` should call `s.set_length(...)` and `s.set_width(...)`, not create a new `Shape`.
</details>

<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

Full solution:

    public static void growShape(Shape s) {
        s.set_length(s.get_length() + 5);
        s.set_width(s.get_width() + 5);
    }

Output:

    rectangle: length=10 width=5
    rectangle: length=15 width=10

Since `s` is an alias of the `Shape` passed in, calling its setters mutates the one shared object. Reading the current value with the getter, adding 5, then writing it back with the setter is what makes the growth visible outside the method once it returns -- the same mutate-through-a-parameter mechanism as the `doubleWidth` example above.

</div>
</details>


## Aliasing

Because two variables can reference the same object, changes made through one variable are visible through the other.

In [ ]:
// CODE_RUNNER: Press Run and see that a and b are aliases of the same Shape -- mutating through b shows up when printing a.

class Shape {
    protected String name;
    private int length;
    private int width;

    public Shape(String name, int length, int width) {
        this.name = name;
        this.length = length;
        this.width = width;
    }

    public void set_width(int w) { this.width = w; }

    public void print_shape() {
        System.out.println(this.name + ": length=" + this.length + " width=" + this.width);
    }
}

public class AliasDemo {
    public static void main(String[] args) {
        Shape a = new Shape("square", 4, 4);
        Shape b = a; // b is now an alias for the same object as a

        b.set_width(99);
        a.print_shape();
    }
}


<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

    square: length=4 width=99

`a` and `b` are two different variable names pointing at *one* `Shape` object -- there is only one object in memory. Mutating it through `b` (`b.set_width(99)`) is indistinguishable from mutating it through `a`, because both variables hold copies of the exact same reference. This is what "aliasing" means: multiple variables, one shared object.

</div>
</details>


## Returning Object References from Methods

This half of the lesson comes from **Topic 5.4, "Accessor Methods"** (Learning Objective MOD-2.D). When a method returns a reference to an object, only a copy of the reference is returned, not the object itself (College Board, 2020, p. 97) - the caller gets a reference to the same object, not a new copy.


In [ ]:
// CODE_RUNNER: Press Run and see that winner is the same object as big -- mutating winner mutates big too.

class Shape {
    protected String name;
    private int length;
    private int width;

    public Shape(String name, int length, int width) {
        this.name = name;
        this.length = length;
        this.width = width;
    }

    public int get_width() { return this.width; }
    public void set_width(int w) { this.width = w; }

    public double calc_area() {
        return this.length * this.width;
    }

    public void print_shape() {
        System.out.println(this.name + ": length=" + this.length + " width=" + this.width);
    }
}

public class ReturnRefDemo {
    public static Shape biggerShape(Shape s1, Shape s2) {
        if (s1.calc_area() >= s2.calc_area()) {
            return s1;
        }
        return s2;
    }

    public static void main(String[] args) {
        Shape small = new Shape("small", 2, 2);
        Shape big = new Shape("big", 10, 10);

        Shape winner = biggerShape(small, big);
        winner.set_width(50);
        big.print_shape();
    }
}


<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

    big: length=10 width=50

`biggerShape` returns `s2` (which is `big`) without copying it -- returning a reference hands back a copy of the *reference*, not the object. So `winner` refers to the exact same object as `big`. Calling `winner.set_width(50)` mutates that shared object, and since `big` points at it too, `big.print_shape()` shows the updated width.

</div>
</details>


## Popcorn Hack 2

> [!TIP]
> Write a method `combine(Shape s1, Shape s2)` that returns a **brand new** `Shape` whose `name` joins `s1`'s and `s2`'s names (e.g. `"square-triangle"`), and whose `length`/`width` are the sums of each shape's length/width.

In [ ]:
// CODE_RUNNER: Fill in combine so it returns a brand new Shape combining s1 and s2, then hit Run to check the output.

class Shape {
    protected String name;
    private int length, width;
    public Shape(String name, int length, int width) { this.name = name; this.length = length; this.width = width; }
    public String get_name() { return name; }
    public int get_length() { return length; }
    public int get_width() { return width; }
    public void set_width(int w) { width = w; }
    public void print_shape() { System.out.println(name + ": length=" + length + " width=" + width); }
}

public class Main {
    // TODO: return a brand new Shape joining s1/s2 names with "-", summing length/width
    public static Shape combine(Shape s1, Shape s2) {
        return null;
    }

    public static void main(String[] args) {
        Shape a = new Shape("square", 4, 4);
        Shape b = new Shape("triangle", 3, 3);
        Shape result = combine(a, b);
        result.print_shape(); // expect: square-triangle: length=7 width=7
        result.set_width(999); // mutating result should NOT affect a or b
        a.print_shape();
        b.print_shape();
    }
}


<details>
<summary>Need a hint?</summary>

Since `combine` builds the result with `new Shape(...)` instead of returning `s1` or `s2` directly, mutating the returned `Shape` afterward should **not** affect `s1` or `s2`. Verify that with a print statement - it's the key difference from `biggerShape` above.
</details>

---

<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

Full solution:

    public static Shape combine(Shape s1, Shape s2) {
        return new Shape(s1.get_name() + "-" + s2.get_name(),
                          s1.get_length() + s2.get_length(),
                          s1.get_width() + s2.get_width());
    }

Output:

    square-triangle: length=7 width=7
    square: length=4 width=4
    triangle: length=3 width=3

Unlike `biggerShape`, `combine` builds a *brand-new* `Shape` with `new` rather than returning `s1` or `s2` directly. Because the returned object is distinct from both inputs, mutating it afterward (`result.set_width(999)`) has no effect on `a` or `b` -- they still reference their own original, untouched objects.

</div>
</details>


## Practice Problems

Three more problems that push past the popcorn hacks above. Try each one on paper (or in a real editor) before revealing the hint.

### Problem 1 - The Swap That Isn't

A classmate writes this method hoping to swap the names of two `Shape`s:

    public static void swapNames(Shape s1, Shape s2) {
        Shape temp = s1;
        s1 = s2;
        s2 = temp;
    }

    Shape a = new Shape("square", 4, 4);
    Shape b = new Shape("triangle", 3, 3);
    swapNames(a, b);
    a.print_shape();
    b.print_shape();

What actually prints, and why doesn't the swap work?

<details>
<summary>Need a hint?</summary>

`swapNames` only reassigns its own local copies of the references (`s1` and `s2`). None of those three assignments ever calls a setter on the actual objects `a` and `b` refer to, so the caller's variables never change. It still prints `square` for `a` and `triangle` for `b` - completely unswapped. To really swap the data, you'd need to mutate fields (e.g. swap `name` through a `set_name` setter), not reassign the parameters.
</details>

<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

It prints:

    square
    triangle

completely unswapped, because `swapNames` only ever reassigns its own local copies `s1` and `s2` -- it never calls a setter on the real objects `a` and `b` refer to. Corrected version, mutating fields instead of reassigning references:

    public static void swapNames(Shape s1, Shape s2) {
        String temp = s1.get_name();
        s1.set_name(s2.get_name());
        s2.set_name(temp);
    }

This works because `set_name` mutates the shared objects through the aliased references -- the same principle as `doubleWidth` earlier in the lesson, just applied to swap two fields instead of growing one.

</div>
</details>


### Problem 2 - Reference Equality vs. Value Equality

Write a method `isSameShape(Shape s1, Shape s2)` that returns `true` only when `s1` and `s2` are aliases of the **exact same object** (not just two objects with equal dimensions). Then test it against three cases: the same reference, two different objects with identical `length`/`width`, and two different objects with different dimensions.

In [ ]:
// CODE_RUNNER: Fill in isSameShape so it only returns true when both parameters are aliases of the exact same object, then hit Run to check all three test cases.

class Shape {
    protected String name;
    private int length;
    private int width;

    public Shape(String name, int length, int width) {
        this.name = name;
        this.length = length;
        this.width = width;
    }

    public int get_length() { return this.length; }
    public int get_width() { return this.width; }
}

public class Main {

    // TODO: return true only when s1 and s2 are aliases of the exact same object
    public static boolean isSameShape(Shape s1, Shape s2) {
        return false;
    }

    public static void main(String[] args) {
        Shape a = new Shape("square", 4, 4);
        Shape b = a;
        Shape c = new Shape("square", 4, 4);
        Shape d = new Shape("triangle", 3, 3);

        System.out.println("a and b (same object): " + isSameShape(a, b));
        System.out.println("a and c (equal dimensions, different object): " + isSameShape(a, c));
        System.out.println("a and d (different object, different dimensions): " + isSameShape(a, d));

        // Expected output:
        // a and b (same object): true
        // a and c (equal dimensions, different object): false
        // a and d (different object, different dimensions): false
    }
}


<details>
<summary>Need a hint?</summary>

Use `==` on the object references, not `.equals(...)`: `return s1 == s2;`. Because `==` on reference types compares the addresses stored in the variables, it's only `true` when both variables are aliases of one object - two separate `Shape`s with matching `length`/`width` still fail this check, since they live at different addresses.
</details>

<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

Full solution:

    public static boolean isSameShape(Shape s1, Shape s2) {
        return s1 == s2;
    }

Output:

    a and b (same object): true
    a and c (equal dimensions, different object): false
    a and d (different object, different dimensions): false

`==` on reference types compares the *addresses* stored in the variables, not the data the objects hold. `a` and `b` are aliases (same address) so `==` is true; `a` and `c` are two separate `Shape` objects that merely happen to hold equal `length`/`width` values, so their addresses differ and `==` is false, even though the data looks identical.

</div>
</details>


### Problem 3 - Find and Fix the Bug

This method is supposed to reset a `Shape` back to a 1×1 square, but callers report the original object never changes:

    public static void resetShape(Shape s) {
        s = new Shape("square", 1, 1);
    }

Fix it so the caller's object is actually reset.

<details>
<summary>Need a hint?</summary>

The bug is the same reassignment pitfall from Problem 1 - `s = new Shape(...)` only repoints the local variable `s`, it never touches the object the caller passed in. The fix is to mutate the existing object's fields instead: `s.set_name("square"); s.set_length(1); s.set_width(1);`.
</details>

---

<details>
<summary><span style="color:#FFD700;">Answer (instructor only)</span></summary>

<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

Corrected version:

    public static void resetShape(Shape s) {
        s.set_name("square");
        s.set_length(1);
        s.set_width(1);
    }

The original bug (`s = new Shape("square", 1, 1);`) only repoints the local parameter `s` at a new object -- it never touches the object the caller passed in, so the caller sees no change once the method returns. Mutating the existing object's fields through its setters, instead of reassigning `s`, is what makes the reset visible to the caller -- the same mutate-vs-reassign distinction that runs through the entire lesson.

</div>
</details>


## Key Takeaways

- Objects are passed by value, but the value being copied is a **reference**, not the object itself.
- Mutating an object's fields through a parameter changes the original object - the caller sees it.
- Reassigning a parameter to a new object only changes the local copy of the reference - the caller's variable is untouched.
- Two variables that reference the same object are **aliases**; mutating through one is visible through the other.
- A method that returns an object reference hands back a reference to the existing object, unless it explicitly builds a new one with `new`.

---

## Check Your Understanding

**Q1.** Inside a method, a parameter `s` calls `s.set_width(99)`. What happens to the caller's object?

A. Nothing - only the local copy changes  
B. It throws a compile error  
C. The caller's object is mutated too, since s is an alias of it  
D. It depends on whether width is public  

---

**Q2.** Inside a method, a parameter `s` is reassigned with `s = new Shape(...);`. What happens to the caller's variable?

A. It now points to the new Shape too  
B. It is unaffected - it still points to the original object  
C. It becomes null  
D. The program crashes  

---

**Q3.** A method does `return s1;` where s1 is a Shape parameter. What does the caller receive?

A. A brand new deep copy of the object  
B. Just the object's name as a String  
C. A reference to the exact same object s1 pointed to  
D. Nothing - non-void methods cannot return objects  


<div markdown="1" style="color:#FFD700; background:rgba(255,215,0,0.07); border-left:3px solid #FFD700; padding:0.75rem 1rem; margin:0.5rem 0;">

**Instructor answer key**

**Q1: C.** The parameter `s` is a copy of the reference, but it still points at the same object as the caller's variable. Calling a setter on `s` mutates that shared object, so the caller sees the change -- this is the core "mutate through a parameter" behavior from section A.

**Q2: B.** Reassigning a parameter only repoints that parameter's *local copy* of the reference. The caller's variable keeps pointing at the original object, since parameters are passed by value -- this is the "reassign does NOT escape" behavior from section B.

**Q3: C.** Returning a reference hands back a *copy of the reference itself*, not a copy of the object it points to -- so the caller receives a reference to the exact same object, matching the `biggerShape`/`combine` distinction covered earlier.

</div>

## References

College Board. (2020). *AP Computer Science A course and exam description* (Unit 5: Writing Classes, Topics 5.4 & 5.6, pp. 97, 100–101). https://apcentral.collegeboard.org/media/pdf/ap-computer-science-a-course-and-exam-description.pdf

Oracle. (n.d.). *Passing reference data type arguments*. The Java Tutorials. Oracle Corporation. Retrieved September 17, 2026, from https://docs.oracle.com/javase/tutorial/java/javaOO/arguments.html